# 🎯 Automated Candidate Interview & Evaluation System

This notebook demonstrates an AI-powered interview system using **AutoGen AgentChat**. Three specialised agents collaborate in a round-robin group chat to conduct a realistic mock interview, collect human answers, and deliver actionable feedback.

| Agent | Role |
|-------|------|
| **Interviewer** | Asks structured interview questions tailored to the role |
| **Candidate** | Human-in-the-loop proxy that accepts real user input |
| **Evaluator** | Career coach that critiques answers and suggests improvements |

## Learning Objectives
In this notebook, you will learn:
1. **Multi-Agent Orchestration** — How to wire up a `RoundRobinGroupChat` with AutoGen AgentChat
2. **Human-in-the-Loop** — Using `UserProxyAgent` to inject real human input into an agent loop
3. **Streaming Results** — Consuming `run_stream` to display interview progress in real time
4. **Termination Conditions** — Controlling conversation length with `TextMentionTermination`

## Prerequisites
- Python 3.10+
- A **Groq API key** stored in a `.env` file (`GROQ_API_KEY=...`)
- Packages: `autogen-agentchat`, `autogen-ext[openai]`, `python-dotenv`

---
## 🔧 Part 1: Environment Setup & Imports

Load environment variables, configure the Groq-hosted LLM client, and import the AutoGen AgentChat primitives we will use throughout the notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports, API Keys & LLM Client
# ============================================================================

import os

from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

# --- Load environment variables from .env ---
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# --- Configure the LLM client (Groq-hosted Llama 3.3 70B) ---
MODEL_NAME = "llama-3.3-70b-versatile"

model_client = OpenAIChatCompletionClient(
    model=MODEL_NAME,
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
    },
)

print(f"✅ Environment loaded — using model: {MODEL_NAME}")

---
## 🤖 Part 2: Define the Interview Agents & Team

We create three agents — **Interviewer**, **Candidate**, and **Evaluator** — and wire them into a `RoundRobinGroupChat`. The conversation ends automatically after the interviewer says *TERMINATE*.

### Key Concepts:
- **AssistantAgent** — LLM-backed agent driven by a system prompt
- **UserProxyAgent** — Bridges human input into the agent loop
- **RoundRobinGroupChat** — Cycles through participants in a fixed order
- **TextMentionTermination** — Stops the chat when a trigger word appears

In [ ]:
# ============================================================================
# AGENT DEFINITIONS: Interviewer, Candidate & Evaluator
# ============================================================================

async def build_interview_team(job_position: str = "AI Engineer"):
    """Create and return a RoundRobinGroupChat team for the given role."""

    # --- Interviewer: asks 3 targeted questions then terminates ---
    interviewer = AssistantAgent(
        name="Interviewer",
        model_client=model_client,
        description=f"An AI agent that conducts interviews for a {job_position} position.",
        system_message=(
            f"You are a professional interviewer for a {job_position} position.\n"
            "Ask one clear question at a time and wait for the user to respond.\n"
            "Ignore the career coach's response — your job is only to ask questions.\n"
            "Tailor each follow-up based on the candidate's previous answer.\n"
            "Ask 3 questions total covering: technical skills, problem-solving, and cultural fit.\n"
            "After the 3rd question, say 'TERMINATE'.\n"
            "Keep every question under 50 words."
        ),
    )

    # --- Candidate: human-in-the-loop proxy ---
    candidate = UserProxyAgent(
        name="Candidate",
        description=f"A human proxy representing the candidate for a {job_position} position.",
        input_func=input,
    )

    # --- Evaluator: career coach giving real-time feedback ---
    evaluator = AssistantAgent(
        name="Evaluator",
        model_client=model_client,
        description=f"An AI career coach providing feedback for a {job_position} interview.",
        system_message=(
            f"You are a career coach specializing in {job_position} interviews.\n"
            "Provide constructive feedback on the candidate's responses and suggest improvements.\n"
            "After the interview, summarize performance and give actionable advice.\n"
            "Keep feedback under 100 words."
        ),
    )

    # --- Assemble the team ---
    termination_condition = TextMentionTermination(text="TERMINATE")

    team = RoundRobinGroupChat(
        participants=[interviewer, candidate, evaluator],
        termination_condition=termination_condition,
        max_turns=20,
    )

    return team

---
## 🔄 Part 3: Stream the Interview

Define an async generator that consumes the team's `run_stream` output and yields formatted messages as they arrive. This keeps the interview interactive rather than blocking until completion.

In [ ]:
# ============================================================================
# INTERVIEW STREAMING: Async Generator for Real-Time Output
# ============================================================================

async def run_interview(team: RoundRobinGroupChat):
    """Stream interview messages, yielding formatted strings as they arrive."""
    async for message in team.run_stream(
        task="Start the interview with the first question."
    ):
        if isinstance(message, TaskResult):
            yield f"✅ Interview completed — stop reason: {message.stop_reason}"
        else:
            yield f"🗣️ {message.source}: {message.content}"

---
## 🚀 Part 4: Run the Interview

Build the team for the desired role and kick off the interactive interview. You will be prompted for input in the terminal each time it is the **Candidate's** turn.

> **Note**: Change `job_position` below to tailor questions to a different role (e.g., *"Data Scientist"*, *"Backend Engineer"*).

In [ ]:
# ============================================================================
# RUN INTERVIEW: Build Team & Stream Conversation
# ============================================================================

job_position = "AI Engineer"
team = await build_interview_team(job_position)

print(f"🤖 Starting interview for: {job_position}\n")

async for message in run_interview(team):
    print("-" * 70)
    print(message)

## 📝 Summary

In this notebook, we built an end-to-end **AI-powered mock interview system** using AutoGen AgentChat.

### 1. Environment Setup
- Loaded a **Groq-hosted Llama 3.3 70B** model via `OpenAIChatCompletionClient`

### 2. Agent Design
- **Interviewer** — asks 3 role-specific questions, then terminates
- **Candidate** — `UserProxyAgent` bridging real human input
- **Evaluator** — career coach delivering constructive feedback after each answer

### 3. Orchestration & Streaming
- `RoundRobinGroupChat` manages turn order across agents
- `TextMentionTermination` ends the loop when *TERMINATE* appears
- `run_stream` provides real-time output instead of blocking

### Next Steps
- Swap `input_func` for a web-based UI (e.g., Gradio, Streamlit) for a richer candidate experience
- Add more agents — e.g., a **Technical Assessor** that scores code-related answers
- Persist transcripts to a database for later review and analytics